In [9]:
import joblib
import numpy as np
import pandas as pd
from interpret.glassbox import ExplainableBoostingClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    brier_score_loss,
    f1_score,
    log_loss,
    roc_auc_score,
)
from sklearn.model_selection import RandomizedSearchCV
from interpret import show
ebm = ExplainableBoostingClassifier()

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [10]:
# Read the Parquet file
df_train = pd.read_parquet("./Paquets/tenant_scoring_train_20260108_164602.parquet", engine="pyarrow")

print("Data train from Parquet file:")
df_train

Data train from Parquet file:


,tenant_id,tenant_type,total_rent,is_particular,is_entreprise,monthly_income,monthly_charge,debt,professional_experience,children_number,...,min_payment_amount_3m,max_payment_amount_3m,payment_success_rate_3m,late_payment_count_3m,avg_days_late_3m,max_days_late_3m,total_late_amount_3m,observation_date,future_invoice_count,target
0,a2538a73-9c86-41d7-af9b-dbddddaca654,Particular,1,1,0,170000.0,444320.0,1635.0,10,4,...,0.0,0.0,0.0,0,0.0,0.0,0.0,2024-06-30,1,1
1,46573e2f-6d89-4361-8e3a-c709b75cc355,Enterprise,1,0,0,195000.0,169745.0,209555.0,17,8,...,8904280.0,8904280.0,1.0,0,0.0,0.0,0.0,2024-10-28,0,0
2,1b97fd90-b375-40d3-9e7f-cd28fd8fadd1,Enterprise,1,0,0,175000.0,149960.0,290275.0,30,6,...,193960.0,193960.0,1.0,0,0.0,0.0,0.0,2024-04-01,0,0
3,b9cd3ab7-2386-4ad9-8023-3e924a7dd129,Enterprise,1,0,0,610000.0,755680.0,664630.0,20,6,...,1102912.0,1102912.0,1.0,0,0.0,0.0,0.0,2024-03-02,0,0
4,da2228b2-aa7d-40a2-ac67-e7c167b771a3,Enterprise,1,0,0,140000.0,122570.0,43655.0,17,5,...,0.0,0.0,0.0,0,0.0,0.0,0.0,2024-05-01,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
595,062d3ed1-0d9c-47cb-b29a-4df1001eb1d5,Enterprise,1,0,0,280000.0,21755.0,293445.0,22,1,...,969800.0,969800.0,1.0,0,0.0,0.0,0.0,2024-09-28,0,0
596,b4e7b437-75a0-4ea1-9439-39c2a7970463,Enterprise,1,0,0,785000.0,2573540.0,705060.0,13,5,...,1433260.0,1433260.0,1.0,0,0.0,0.0,0.0,2024-09-28,0,0
597,00815d48-bfd7-405f-9f99-3d12a4d7b1d6,Enterprise,1,0,0,200000.0,574170.0,85525.0,22,5,...,732410.0,732410.0,1.0,0,0.0,0.0,0.0,2024-08-29,0,0
598,5db2349f-1b37-4887-9e69-996239ec5863,Enterprise,1,0,0,480000.0,1245275.0,553300.0,0,6,...,808833.0,808833.0,1.0,0,0.0,0.0,0.0,2024-03-02,0,0


In [6]:
# Read the Parquet file
df_test = pd.read_parquet("./Paquets/tenant_scoring_test_20260108_164602.parquet", engine="pyarrow")

print("Data test from Parquet file:")
# df_test.columns

Data test from Parquet file:


In [7]:
# Read the Parquet file
df_validation = pd.read_parquet("./Paquets/tenant_scoring_validation_20260108_164602.parquet", engine="pyarrow")

print("Data validation from Parquet file:")
# df_validation.columns

Data validation from Parquet file:


In [8]:
# Read the Parquet file
df_calibration = pd.read_parquet("./Paquets/tenant_scoring_calibration_20260108_164602.parquet", engine="pyarrow")

print("Data calibration from Parquet file:")
# df_calibration.columns

Data calibration from Parquet file:


In [12]:
df_train.info(verbose=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 600 entries, 0 to 599
Data columns (total 38 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   tenant_id                600 non-null    object        
 1   tenant_type              600 non-null    object        
 2   total_rent               600 non-null    int64         
 3   is_particular            600 non-null    int32         
 4   is_entreprise            600 non-null    int32         
 5   monthly_income           600 non-null    float64       
 6   monthly_charge           600 non-null    float64       
 7   debt                     600 non-null    float64       
 8   professional_experience  600 non-null    int64         
 9   children_number          600 non-null    int64         
 10  persons_dependents       600 non-null    int64         
 11  contract_type            600 non-null    object        
 12  occupational_function    600 non-nul

In [13]:
df_train.describe()

,total_rent,is_particular,is_entreprise,monthly_income,monthly_charge,debt,professional_experience,children_number,persons_dependents,lease_amount,...,min_payment_amount_3m,max_payment_amount_3m,payment_success_rate_3m,late_payment_count_3m,avg_days_late_3m,max_days_late_3m,total_late_amount_3m,observation_date,future_invoice_count,target
count,600.0,600.000000,600.0,6.000000e+02,6.000000e+02,6.000000e+02,600.000000,600.000000,600.000000,600.0,...,6.000000e+02,6.000000e+02,600.000000,600.0,600.0,600.0,600.0,600,600.000000,600.000000
mean,1.0,0.400000,0.0,3.572333e+05,5.784606e+05,3.916112e+05,20.331667,4.768333,3.428333,0.0,...,1.124144e+06,1.124144e+06,0.595000,0.0,0.0,0.0,0.0,2024-06-10 15:36:00,0.296667,0.296667
min,1.0,0.000000,0.0,7.500000e+04,2.175500e+04,1.635000e+03,0.000000,0.000000,0.000000,0.0,...,0.000000e+00,0.000000e+00,0.000000,0.0,0.0,0.0,0.0,2024-02-01 00:00:00,0.000000,0.000000
25%,1.0,0.000000,0.0,2.187500e+05,2.002600e+05,1.470250e+05,13.000000,1.000000,2.000000,0.0,...,0.000000e+00,0.000000e+00,0.000000,0.0,0.0,0.0,0.0,2024-04-01 00:00:00,0.000000,0.000000
50%,1.0,0.000000,0.0,2.900000e+05,4.443200e+05,3.420300e+05,23.000000,5.000000,3.000000,0.0,...,1.470140e+05,1.470140e+05,1.000000,0.0,0.0,0.0,0.0,2024-05-31 00:00:00,0.000000,0.000000
75%,1.0,1.000000,0.0,4.500000e+05,7.804650e+05,4.893088e+05,29.000000,8.000000,6.000000,0.0,...,9.162875e+05,9.162875e+05,1.000000,0.0,0.0,0.0,0.0,2024-08-29 00:00:00,1.000000,1.000000
max,1.0,1.000000,0.0,1.400000e+06,3.323235e+06,2.791645e+06,35.000000,10.000000,7.000000,0.0,...,1.195632e+07,1.195632e+07,1.000000,0.0,0.0,0.0,0.0,2024-10-28 00:00:00,1.000000,1.000000
std,0.0,0.490307,0.0,2.263140e+05,5.185802e+05,3.861034e+05,9.749877,3.379403,2.249436,0.0,...,2.467042e+06,2.467042e+06,0.491302,0.0,0.0,0.0,0.0,NaN,0.457169,0.457169


In [14]:
df_train.isna().sum()

tenant_id                    0
tenant_type                  0
total_rent                   0
is_particular                0
is_entreprise                0
monthly_income               0
monthly_charge               0
debt                         0
professional_experience      0
children_number              0
persons_dependents           0
contract_type                0
occupational_function        0
lease_amount                 0
lease_status                 0
pay_day                      0
penalty                      0
rental_charge                0
fees_included              600
lease_duration_days          0
lease_is_active              0
fees_included_flag           0
total_amount_owed            0
has_had_penalties            0
payment_count_3m             0
avg_payment_amount_3m        0
total_payment_amount_3m      0
payment_consistency_3m       0
min_payment_amount_3m        0
max_payment_amount_3m        0
payment_success_rate_3m      0
late_payment_count_3m        0
avg_days

In [15]:
missing_counts = df_train.isnull().sum()
missing_counts = missing_counts[missing_counts > 0]

print("Variables avec valeurs manquantes :")
print(missing_counts)

Variables avec valeurs manquantes :
fees_included    600
dtype: int64


In [17]:
df_train.nunique()

tenant_id                  100
tenant_type                  2
total_rent                   1
is_particular                2
is_entreprise                1
monthly_income              59
monthly_charge             100
debt                       100
professional_experience     33
children_number             11
persons_dependents           8
contract_type                7
occupational_function       99
lease_amount                 1
lease_status                 1
pay_day                      1
penalty                      1
rental_charge                1
fees_included                0
lease_duration_days          1
lease_is_active              1
fees_included_flag           1
total_amount_owed            1
has_had_penalties            1
payment_count_3m             3
avg_payment_amount_3m       57
total_payment_amount_3m     30
payment_consistency_3m       1
min_payment_amount_3m       57
max_payment_amount_3m       57
payment_success_rate_3m      2
late_payment_count_3m        1
avg_days

In [18]:
df_train.columns

Index(['tenant_id', 'tenant_type', 'total_rent', 'is_particular',
       'is_entreprise', 'monthly_income', 'monthly_charge', 'debt',
       'professional_experience', 'children_number', 'persons_dependents',
       'contract_type', 'occupational_function', 'lease_amount',
       'lease_status', 'pay_day', 'penalty', 'rental_charge', 'fees_included',
       'lease_duration_days', 'lease_is_active', 'fees_included_flag',
       'total_amount_owed', 'has_had_penalties', 'payment_count_3m',
       'avg_payment_amount_3m', 'total_payment_amount_3m',
       'payment_consistency_3m', 'min_payment_amount_3m',
       'max_payment_amount_3m', 'payment_success_rate_3m',
       'late_payment_count_3m', 'avg_days_late_3m', 'max_days_late_3m',
       'total_late_amount_3m', 'observation_date', 'future_invoice_count',
       'target'],
      dtype='object')

In [19]:
df_train

,tenant_id,tenant_type,total_rent,is_particular,is_entreprise,monthly_income,monthly_charge,debt,professional_experience,children_number,...,min_payment_amount_3m,max_payment_amount_3m,payment_success_rate_3m,late_payment_count_3m,avg_days_late_3m,max_days_late_3m,total_late_amount_3m,observation_date,future_invoice_count,target
0,a2538a73-9c86-41d7-af9b-dbddddaca654,Particular,1,1,0,170000.0,444320.0,1635.0,10,4,...,0.0,0.0,0.0,0,0.0,0.0,0.0,2024-06-30,1,1
1,46573e2f-6d89-4361-8e3a-c709b75cc355,Enterprise,1,0,0,195000.0,169745.0,209555.0,17,8,...,8904280.0,8904280.0,1.0,0,0.0,0.0,0.0,2024-10-28,0,0
2,1b97fd90-b375-40d3-9e7f-cd28fd8fadd1,Enterprise,1,0,0,175000.0,149960.0,290275.0,30,6,...,193960.0,193960.0,1.0,0,0.0,0.0,0.0,2024-04-01,0,0
3,b9cd3ab7-2386-4ad9-8023-3e924a7dd129,Enterprise,1,0,0,610000.0,755680.0,664630.0,20,6,...,1102912.0,1102912.0,1.0,0,0.0,0.0,0.0,2024-03-02,0,0
4,da2228b2-aa7d-40a2-ac67-e7c167b771a3,Enterprise,1,0,0,140000.0,122570.0,43655.0,17,5,...,0.0,0.0,0.0,0,0.0,0.0,0.0,2024-05-01,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
595,062d3ed1-0d9c-47cb-b29a-4df1001eb1d5,Enterprise,1,0,0,280000.0,21755.0,293445.0,22,1,...,969800.0,969800.0,1.0,0,0.0,0.0,0.0,2024-09-28,0,0
596,b4e7b437-75a0-4ea1-9439-39c2a7970463,Enterprise,1,0,0,785000.0,2573540.0,705060.0,13,5,...,1433260.0,1433260.0,1.0,0,0.0,0.0,0.0,2024-09-28,0,0
597,00815d48-bfd7-405f-9f99-3d12a4d7b1d6,Enterprise,1,0,0,200000.0,574170.0,85525.0,22,5,...,732410.0,732410.0,1.0,0,0.0,0.0,0.0,2024-08-29,0,0
598,5db2349f-1b37-4887-9e69-996239ec5863,Enterprise,1,0,0,480000.0,1245275.0,553300.0,0,6,...,808833.0,808833.0,1.0,0,0.0,0.0,0.0,2024-03-02,0,0


In [20]:
df_train[0]

KeyError: 0

In [23]:
#fais sortir les données de un seul utilisateur
df_train_user = df_train[df_train['tenant_id'] == df_train['tenant_id'].iloc[0]]
df_train_user

,tenant_id,tenant_type,total_rent,is_particular,is_entreprise,monthly_income,monthly_charge,debt,professional_experience,children_number,...,min_payment_amount_3m,max_payment_amount_3m,payment_success_rate_3m,late_payment_count_3m,avg_days_late_3m,max_days_late_3m,total_late_amount_3m,observation_date,future_invoice_count,target
0,a2538a73-9c86-41d7-af9b-dbddddaca654,Particular,1,1,0,170000.0,444320.0,1635.0,10,4,...,0.0,0.0,0.0,0,0.0,0.0,0.0,2024-06-30,1,1
36,a2538a73-9c86-41d7-af9b-dbddddaca654,Particular,1,1,0,170000.0,444320.0,1635.0,10,4,...,131491.0,131491.0,1.0,0,0.0,0.0,0.0,2024-02-01,0,0
84,a2538a73-9c86-41d7-af9b-dbddddaca654,Particular,1,1,0,170000.0,444320.0,1635.0,10,4,...,0.0,0.0,0.0,0,0.0,0.0,0.0,2024-05-31,1,1
300,a2538a73-9c86-41d7-af9b-dbddddaca654,Particular,1,1,0,170000.0,444320.0,1635.0,10,4,...,0.0,0.0,0.0,0,0.0,0.0,0.0,2024-07-30,1,1


In [25]:
#enregistrer le dataframe filtré dans un nouveau fichier list
df_train.to_csv('df_train_user.csv', index=False)

In [ ]:
# Fit the model to your data (assuming X_train and y_train are defined)
# ebm.fit(X_train, y_train)

# Get global explanations and show visualizations
# global_explanation = ebm.explain_global()
# show(global_explanation)